# exp_7 · 03 decision — H1 and report tables (IDSSE only)
H1 = on provider **and** `L_ctrl`: `exp7_robust_ens3` measured − `exp3_ref_ens3` `target_like` (exp_3's stored rows)
paired 95 % CI low > 0; `target_like` mean Δ ≥ −0.005; stress mean Δ ≥ −0.005. Measured and stress use the
measured-view selection; `target_like` uses the `target_like` selection. Results: `results/decision.json`.

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp7_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_7')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp7_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_03.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_6 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp7_decision as DE
import exp7_eval as E
import exp7_report as R

PREREG verified 189874eb7d34ab4c | frozen 2026-11-11T01:29:38+00:00


nothing written under exp_0-exp_6 / data since freezing: True {'exp_0': True, 'exp_1': True, 'exp_2': True, 'exp_3': True, 'exp_4': True, 'exp_5': True, 'exp_6': True, 'data': True, 'data_metric_repo': True}


In [2]:
results = {(sel, label): C1.read_json(E.result_path(C.PRIMARY, sel, label))
           for sel in ('target_like', 'measured') for label in C.LABEL_SETS}
deltas = {label: {'gain': DE.view_deltas(results[('measured', label)]['rows'], 'measured', label),
                  'no_harm': DE.view_deltas(results[('target_like', label)]['rows'], 'target_like', label),
                  'stress': DE.view_deltas(results[('measured', label)]['rows'], 'stress', label)}
          for label in C.LABEL_SETS}
decision = DE.decide(DE.clause_table(deltas))
decision.update({'decided_at_utc': now(), 'primary': C.PRIMARY, 'baseline': DE.H1_RULE['baseline'],
                 'final_configs': {label: results[('measured', label)]['final_config'] for label in C.LABEL_SETS},
                 'files_written': False,
                 'candidate_files': 'written by 04_target' if decision['h1_passed'] else 'none: H1 failed'})
C.write_results(C.RESULTS / 'decision.json', decision)
for label in C.LABEL_SETS:
    for clause, value in decision['clauses'][label].items():
        print(label, clause, {k: (round(value[k], 4) if isinstance(value[k], float) else value[k])
                              for k in ('mean_delta', 'ci_low', 'ci_high', 'wins', 'pass')})
print('H1 passed:', decision['h1_passed'], '| failed:', decision['failed'])

provider gain {'mean_delta': 0.0137, 'ci_low': 0.0033, 'ci_high': 0.0265, 'wins': 6, 'pass': True}
provider no_harm {'mean_delta': -0.0007, 'ci_low': -0.0076, 'ci_high': 0.007, 'wins': 3, 'pass': True}
provider stress {'mean_delta': -0.0041, 'ci_low': -0.009, 'ci_high': 0.0, 'wins': 2, 'pass': True}
ctrl gain {'mean_delta': 0.0022, 'ci_low': -0.0112, 'ci_high': 0.0139, 'wins': 4, 'pass': False}
ctrl no_harm {'mean_delta': -0.0059, 'ci_low': -0.0171, 'ci_high': 0.0038, 'wins': 4, 'pass': False}
ctrl stress {'mean_delta': -0.0054, 'ci_low': -0.0162, 'ci_high': 0.0057, 'wins': 3, 'pass': False}
H1 passed: False | failed: ['gain:ctrl', 'no_harm:ctrl', 'stress:ctrl']


In [3]:
frame = R.all_rows()
rows = R.delta_rows(frame)
spread = R.seed_spread(rows)
C.write_results(C.RESULTS / 'summary.json', {'computed_at_utc': now(), 'deltas': rows, 'seed_spread': spread})
(C.RESULTS / 'report_tables.md').write_text('## Views vs exp3_ref_ens3 target_like\n\n' + R.markdown(rows)
                                            + '\n## Seed spread\n\n' + R.markdown(spread))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(pd.DataFrame(spread).round(4).to_string(index=False))

           variant          selection_view        view label_set  spell_f1  macro_f1  count_error  mean_delta  ci_low  ci_high  wins
  exp7_robust_ens3             target_like target_like  provider    0.6634    0.9188      -0.2163     -0.0007 -0.0076   0.0070     3
  exp7_robust_ens3             target_like target_like      ctrl    0.7424    0.9207      -0.1145     -0.0059 -0.0171   0.0038     4
  exp7_robust_ens3                measured    measured  provider    0.6779    0.9255      -0.2101      0.0137  0.0033   0.0265     6
  exp7_robust_ens3                measured      stress  provider    0.6600    0.9173      -0.2111     -0.0041 -0.0090   0.0000     2
  exp7_robust_ens3                measured         ar1  provider    0.6781    0.9247      -0.2134      0.0140  0.0051   0.0252     6
  exp7_robust_ens3                measured    measured      ctrl    0.7505    0.9268      -0.1023      0.0022 -0.0112   0.0139     4
  exp7_robust_ens3                measured      stress      ctrl    0

In [4]:
final = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / 'guard_final.json')
print('final write guard unchanged:', final['all_unchanged'])

final write guard unchanged: True
